# AI_Agent_Workshop_Day2_01 — Problem Setup and Data

This notebook introduces the municipal challenge for Day 2:

> Build an AI-powered assistant that helps residents understand which level of government is responsible for a service and what to do next.

We will start with a compact, curated dataset and a reproducible schema before building the agent.

## Learning goals

By the end of this notebook, students should be able to:

1. Frame the service-routing problem as a structured AI task.
2. Define an output schema for a public-service assistant.
3. Inspect and normalize a starter service catalog.
4. Export clean artifacts for later retrieval and evaluation.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path("day2_workshop")

DATA_DIR = ROOT / "data"
EVAL_DIR = ROOT / "eval"
ARTIFACTS_DIR = ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

print("Root:", ROOT.resolve())
print("Data files:", [p.name for p in DATA_DIR.iterdir()])

Root: D:\Desktop\8010\AI_Agent_Workshop
Data files: ['service_catalog.csv', 'service_catalog.json']


## The challenge as a machine learning task

A resident question becomes an input example.

We want the assistant to return:

- the most likely service name,
- the responsible level of government,
- the likely responsible body,
- a concise explanation,
- and the next steps.

This is a hybrid task involving semantic matching, classification, retrieval, and answer generation.

In [2]:
response_schema = {
    "service_name": "string",
    "jurisdiction_level": "City | Region | Province | Federal | Mixed | Unclear",
    "responsible_body": "string",
    "confidence": "float in [0, 1]",
    "reasoning_summary": "short grounded explanation",
    "next_steps": ["step 1", "step 2"],
    "sources": ["url 1", "url 2"]
}
response_schema

{'service_name': 'string',
 'jurisdiction_level': 'City | Region | Province | Federal | Mixed | Unclear',
 'responsible_body': 'string',
 'confidence': 'float in [0, 1]',
 'reasoning_summary': 'short grounded explanation',
 'next_steps': ['step 1', 'step 2'],
 'sources': ['url 1', 'url 2']}

## Load the starter catalog

In [3]:
catalog = pd.read_csv(DATA_DIR / "service_catalog.csv")
catalog

,service_name,jurisdiction_level,responsible_body,description,keywords,next_steps_hint,source_url
0,garbage pickup,Region,Region of Waterloo Waste Management,"Residential garbage collection schedule, misse...",garbage;waste;pickup;missed collection;curbside,Check the regional waste collection schedule; ...,https://www.regionofwaterloo.ca/
1,recycling collection,Region,Region of Waterloo Waste Management,Blue box recycling collection and schedule inf...,recycling;blue box;collection,Check the regional collection calendar and rec...,https://www.regionofwaterloo.ca/
2,property tax billing,City,City of Kitchener Revenue Division,"Property tax billing, payment options, and acc...",property tax;tax bill;billing;assessment,Visit the city's property tax page or contact ...,https://www.kitchener.ca/
3,water billing,City,City of Kitchener Utilities,Water billing and account inquiries for city u...,water bill;utilities;billing,Check your city utility account and payment op...,https://www.kitchener.ca/
4,snow removal on regional roads,Region,Region of Waterloo Transportation Services,Maintenance and snow clearing for regional roads.,snow removal;plowing;regional roads;winter mai...,Confirm whether the road is regional before co...,https://www.regionofwaterloo.ca/
5,snow removal on city streets,City,City of Kitchener Operations,Maintenance and snow clearing for local city s...,snow removal;city streets;plowing;roads,Check the city operations page for winter main...,https://www.kitchener.ca/
6,child care subsidies,Province,Government of Ontario,Provincial child care funding and policy frame...,childcare;daycare;subsidy;ontario,Review Ontario child care support information ...,https://www.ontario.ca/
7,public health inspections,Region,Region of Waterloo Public Health,Food premise inspections and public health rep...,public health;inspection;restaurant inspection...,Search the Region's public health resources or...,https://www.regionofwaterloo.ca/
8,employment insurance,Federal,Government of Canada,"Employment Insurance benefits, eligibility, an...",employment insurance;EI;benefits,Use the federal service portal and application...,https://www.canada.ca/
9,passport services,Federal,Government of Canada,"Passport issuance, renewal, and service standa...",passport;travel document;renewal,Use Service Canada passport guidance,https://www.canada.ca/


In [4]:
catalog.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   service_name        15 non-null     str  
 1   jurisdiction_level  15 non-null     str  
 2   responsible_body    15 non-null     str  
 3   description         15 non-null     str  
 4   keywords            15 non-null     str  
 5   next_steps_hint     15 non-null     str  
 6   source_url          15 non-null     str  
dtypes: str(7)
memory usage: 972.0 bytes


In [5]:
catalog.sample(min(len(catalog), 5), random_state=42)

,service_name,jurisdiction_level,responsible_body,description,keywords,next_steps_hint,source_url
9,passport services,Federal,Government of Canada,"Passport issuance, renewal, and service standa...",passport;travel document;renewal,Use Service Canada passport guidance,https://www.canada.ca/
11,road pothole on local street,City,City of Kitchener Transportation Services,Reporting potholes or road defects on local ci...,pothole;road repair;local street,Report the issue through the city's service ch...,https://www.kitchener.ca/
0,garbage pickup,Region,Region of Waterloo Waste Management,"Residential garbage collection schedule, misse...",garbage;waste;pickup;missed collection;curbside,Check the regional waste collection schedule; ...,https://www.regionofwaterloo.ca/
13,income tax filing,Federal,Canada Revenue Agency,Personal income tax filing and account support.,income tax;CRA;tax return,Use CRA account and filing guidance,https://www.canada.ca/
5,snow removal on city streets,City,City of Kitchener Operations,Maintenance and snow clearing for local city s...,snow removal;city streets;plowing;roads,Check the city operations page for winter main...,https://www.kitchener.ca/


## Normalize the service catalog

We will add a few cleaned fields that make retrieval easier later.

In [6]:
def normalize_catalog(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["service_name_normalized"] = out["service_name"].str.lower().str.strip()
    out["keywords_list"] = out["keywords"].fillna("").apply(
        lambda x: [k.strip().lower() for k in x.split(";") if k.strip()]
    )
    out["retrieval_text"] = (
        out["service_name"].fillna("") + " | " +
        out["description"].fillna("") + " | " +
        out["keywords"].fillna("")
    ).str.lower()
    return out

clean_catalog = normalize_catalog(catalog)
clean_catalog.head()

,service_name,jurisdiction_level,responsible_body,description,keywords,next_steps_hint,source_url,service_name_normalized,keywords_list,retrieval_text
0,garbage pickup,Region,Region of Waterloo Waste Management,"Residential garbage collection schedule, misse...",garbage;waste;pickup;missed collection;curbside,Check the regional waste collection schedule; ...,https://www.regionofwaterloo.ca/,garbage pickup,"[garbage, waste, pickup, missed collection, cu...",garbage pickup | residential garbage collectio...
1,recycling collection,Region,Region of Waterloo Waste Management,Blue box recycling collection and schedule inf...,recycling;blue box;collection,Check the regional collection calendar and rec...,https://www.regionofwaterloo.ca/,recycling collection,"[recycling, blue box, collection]",recycling collection | blue box recycling coll...
2,property tax billing,City,City of Kitchener Revenue Division,"Property tax billing, payment options, and acc...",property tax;tax bill;billing;assessment,Visit the city's property tax page or contact ...,https://www.kitchener.ca/,property tax billing,"[property tax, tax bill, billing, assessment]","property tax billing | property tax billing, p..."
3,water billing,City,City of Kitchener Utilities,Water billing and account inquiries for city u...,water bill;utilities;billing,Check your city utility account and payment op...,https://www.kitchener.ca/,water billing,"[water bill, utilities, billing]",water billing | water billing and account inqu...
4,snow removal on regional roads,Region,Region of Waterloo Transportation Services,Maintenance and snow clearing for regional roads.,snow removal;plowing;regional roads;winter mai...,Confirm whether the road is regional before co...,https://www.regionofwaterloo.ca/,snow removal on regional roads,"[snow removal, plowing, regional roads, winter...",snow removal on regional roads | maintenance a...


## Inspect jurisdictional coverage

In [7]:
clean_catalog[["service_name", "jurisdiction_level", "responsible_body"]].sort_values(
    by=["jurisdiction_level", "service_name"]
)

,service_name,jurisdiction_level,responsible_body
2,property tax billing,City,City of Kitchener Revenue Division
11,road pothole on local street,City,City of Kitchener Transportation Services
5,snow removal on city streets,City,City of Kitchener Operations
3,water billing,City,City of Kitchener Utilities
8,employment insurance,Federal,Government of Canada
13,income tax filing,Federal,Canada Revenue Agency
9,passport services,Federal,Government of Canada
10,public transit,Mixed,Grand River Transit / Regional municipality co...
6,child care subsidies,Province,Government of Ontario
14,drivers licence renewal,Province,Government of Ontario / ServiceOntario


## Export cleaned artifacts

In [8]:
output_path = ARTIFACTS_DIR / "service_catalog.cleaned.json"
clean_catalog.to_json(output_path, orient="records", indent=2)
print("Saved:", output_path.resolve())

Saved: D:\Desktop\8010\AI_Agent_Workshop\artifacts\service_catalog.cleaned.json


## Evaluation set preview

In [9]:
eval_df = pd.read_csv(EVAL_DIR / "service_eval_set.csv")
eval_df

,question,expected_service_name,expected_jurisdiction_level,expected_responsible_body
0,Who do I contact about garbage pickup?,garbage pickup,Region,Region of Waterloo Waste Management
1,Is childcare a city or provincial service?,child care subsidies,Province,Government of Ontario
2,Who handles my property tax bill in Kitchener?,property tax billing,City,City of Kitchener Revenue Division
3,How do I renew my driver's licence?,drivers licence renewal,Province,Government of Ontario / ServiceOntario
4,Where do I apply for EI?,employment insurance,Federal,Government of Canada
5,I found a pothole on my street. Who fixes it?,road pothole on local street,City,City of Kitchener Transportation Services
6,Who clears snow on regional roads?,snow removal on regional roads,Region,Region of Waterloo Transportation Services
7,Who do I call about restaurant inspections?,public health inspections,Region,Region of Waterloo Public Health


## Exercise

1. Add 3 more services to the catalog.
2. Create at least 2 ambiguous examples.
3. Mark one example as `Unclear` or `Mixed`.
4. Re-export the cleaned catalog.

### Exercise 1: Add 3 More Services to the Catalog

In this step, we add three new public services to the starter catalog.
These examples help expand the assistant’s knowledge base and improve routing coverage.
The new services include different levels of government responsibility.

In [10]:
# Create 3 new service records to expand the catalog
new_services = pd.DataFrame([
    {
        "service_name": "Snow removal",
        "jurisdiction_level": "City",
        "responsible_body": "City of Kitchener",
        "description": "Snow clearing and winter road maintenance within the city.",
        "keywords": "snow; winter; roads; plowing"
    },
    {
        "service_name": "Passport renewal",
        "jurisdiction_level": "Federal",
        "responsible_body": "Government of Canada",
        "description": "Passport application and renewal services for Canadian residents.",
        "keywords": "passport; renewal; travel; federal"
    },
    {
        "service_name": "OHIP card",
        "jurisdiction_level": "Province",
        "responsible_body": "Government of Ontario",
        "description": "Ontario health card services including renewal and updates.",
        "keywords": "ohip; health card; medical; ontario"
    }
])

# Append the new services to the original catalog
catalog_extended = pd.concat([catalog, new_services], ignore_index=True)

# Preview the newly added rows
catalog_extended.tail(3)

,service_name,jurisdiction_level,responsible_body,description,keywords,next_steps_hint,source_url
15,Snow removal,City,City of Kitchener,Snow clearing and winter road maintenance with...,snow; winter; roads; plowing,NaN,NaN
16,Passport renewal,Federal,Government of Canada,Passport application and renewal services for ...,passport; renewal; travel; federal,NaN,NaN
17,OHIP card,Province,Government of Ontario,Ontario health card services including renewal...,ohip; health card; medical; ontario,NaN,NaN


### Exercise 2: Create 2 Ambiguous Examples

In this step, we add two ambiguous services.
These examples are useful because some resident questions may match multiple government levels or require clarification.
This helps prepare the assistant for more realistic user queries.

In [11]:
# Create 2 ambiguous service examples
ambiguous_services = pd.DataFrame([
    {
        "service_name": "Road maintenance",
        "jurisdiction_level": "Mixed",
        "responsible_body": "City / Region of Waterloo",
        "description": "Road issues may belong to the city or the region depending on the road.",
        "keywords": "road; repair; maintenance; street"
    },
    {
        "service_name": "Housing support",
        "jurisdiction_level": "Unclear",
        "responsible_body": "Various agencies",
        "description": "Housing support programs may depend on the type of service and user situation.",
        "keywords": "housing; shelter; support; assistance"
    }
])

# Append the ambiguous services to the extended catalog
catalog_extended = pd.concat([catalog_extended, ambiguous_services], ignore_index=True)

# Preview the ambiguous examples
catalog_extended.tail(2)

,service_name,jurisdiction_level,responsible_body,description,keywords,next_steps_hint,source_url
18,Road maintenance,Mixed,City / Region of Waterloo,Road issues may belong to the city or the regi...,road; repair; maintenance; street,NaN,NaN
19,Housing support,Unclear,Various agencies,Housing support programs may depend on the typ...,housing; shelter; support; assistance,NaN,NaN


### Exercise 3: Mark One Example as Unclear or Mixed

In this step, we ensure that at least one example is labeled as `Unclear` or `Mixed`.
This is important because not every public service request has a single clear jurisdiction.
It helps the assistant learn when clarification may be needed.

In [12]:
# Check the rows that were labeled as Mixed or Unclear
catalog_extended[
    catalog_extended["jurisdiction_level"].isin(["Mixed", "Unclear"])
][["service_name", "jurisdiction_level", "responsible_body"]]

,service_name,jurisdiction_level,responsible_body
10,public transit,Mixed,Grand River Transit / Regional municipality co...
18,Road maintenance,Mixed,City / Region of Waterloo
19,Housing support,Unclear,Various agencies


### Exercise 4: Re-export the Cleaned Catalog

In this final step, we normalize the updated catalog and export it again as a cleaned JSON artifact.
This file will be used later for retrieval and evaluation in the agent workflow.

In [13]:
# Normalize the updated catalog using the existing helper function
clean_catalog_updated = normalize_catalog(catalog_extended)

# Define the output path for the updated cleaned catalog
updated_output_path = ARTIFACTS_DIR / "service_catalog.cleaned.updated.json"

# Export the cleaned catalog to JSON format
clean_catalog_updated.to_json(updated_output_path, orient="records", indent=2)

# Print the saved file path
print("Updated cleaned catalog saved to:", updated_output_path.resolve())

# Preview the last few rows of the cleaned catalog
clean_catalog_updated.tail()

Updated cleaned catalog saved to: D:\Desktop\8010\AI_Agent_Workshop\artifacts\service_catalog.cleaned.updated.json


,service_name,jurisdiction_level,responsible_body,description,keywords,next_steps_hint,source_url,service_name_normalized,keywords_list,retrieval_text
15,Snow removal,City,City of Kitchener,Snow clearing and winter road maintenance with...,snow; winter; roads; plowing,NaN,NaN,snow removal,"[snow, winter, roads, plowing]",snow removal | snow clearing and winter road m...
16,Passport renewal,Federal,Government of Canada,Passport application and renewal services for ...,passport; renewal; travel; federal,NaN,NaN,passport renewal,"[passport, renewal, travel, federal]",passport renewal | passport application and re...
17,OHIP card,Province,Government of Ontario,Ontario health card services including renewal...,ohip; health card; medical; ontario,NaN,NaN,ohip card,"[ohip, health card, medical, ontario]",ohip card | ontario health card services inclu...
18,Road maintenance,Mixed,City / Region of Waterloo,Road issues may belong to the city or the regi...,road; repair; maintenance; street,NaN,NaN,road maintenance,"[road, repair, maintenance, street]",road maintenance | road issues may belong to t...
19,Housing support,Unclear,Various agencies,Housing support programs may depend on the typ...,housing; shelter; support; assistance,NaN,NaN,housing support,"[housing, shelter, support, assistance]",housing support | housing support programs may...


In [14]:
# Verify the newly added services in the cleaned catalog
clean_catalog_updated[
    clean_catalog_updated["service_name"].isin([
        "Snow removal",
        "Passport renewal",
        "OHIP card",
        "Road maintenance",
        "Housing support"
    ])
][["service_name", "jurisdiction_level", "responsible_body", "keywords_list"]]

,service_name,jurisdiction_level,responsible_body,keywords_list
15,Snow removal,City,City of Kitchener,"[snow, winter, roads, plowing]"
16,Passport renewal,Federal,Government of Canada,"[passport, renewal, travel, federal]"
17,OHIP card,Province,Government of Ontario,"[ohip, health card, medical, ontario]"
18,Road maintenance,Mixed,City / Region of Waterloo,"[road, repair, maintenance, street]"
19,Housing support,Unclear,Various agencies,"[housing, shelter, support, assistance]"


## Reflection

- Which parts of this problem are classification problems?
- Which parts require retrieval?
- Which parts benefit from tool use?
- Which outputs should be deterministic and machine-readable?

## Reflection

### 1. Which parts of this problem are classification problems?

Classification is used to predict the correct jurisdiction level for a service request, such as City, Region, Province, Federal, Mixed, or Unclear.  
It can also classify user intent based on the question they ask.

### 2. Which parts require retrieval?

Retrieval is needed when the assistant must search service details such as department names, contact information, descriptions, policies, or next steps from the catalog or external data sources.

### 3. Which parts benefit from tool use?

Tool use is helpful for accessing updated municipal websites, checking live service status, retrieving records, searching databases, and routing requests automatically.

### 4. Which outputs should be deterministic and machine-readable?

Outputs such as jurisdiction labels, service IDs, contact information, routing decisions, and JSON responses should be deterministic and machine-readable so they can be used reliably in downstream systems.